# 🚀 Build with Gemma Hackathon: Starter Template
Welcome to the official GDG on Campus UoN starter template! This notebook is pre-configured with GPU acceleration and mounts the Gemma model weights directly to the cloud environment so you don't have to waste time downloading files.

### ⚙️ How to use this template:
1. Click **"Copy and Edit"** in the top right corner of this notebook to create your own private version.
2. Ensure your **Accelerator** in the right-hand panel is set to **GPU T4 x2**.
3. Run the cells below to load the model and start building your prototype!

## ⚠️ CRUCIAL STEP BEFORE RUNNING CODE ⚠️
This notebook **REQUIRES** a GPU to load the Gemma model. It will crash on CPU.

### How to activate your free cloud GPU:
1. Look at the top panel under the **title** of the notebook -> **Settings**.
2. Click on **Accelerator**.
3. Select **GPU T4 x2** and click **"Turn on GPU T4 x2"**.

In [ ]:
!pip install -q -U git+https://github.com/huggingface/transformers.git accelerate bitsandbytes

In [ ]:
import kagglehub
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

# Fail-safe hardware verification
if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ HARDWARE ERROR: GPU accelerator is not enabled! "
        "Please go to the right-hand panel, click 'Accelerator', "
        "and switch it to GPU T4 x2 before continuing."
    )

print("⏳ Resolving model files via kagglehub...")
# kagglehub resolves the exact local file path directory instantly
MODEL_PATH = kagglehub.model_download("google/gemma-4/transformers/gemma-4-12b-it")

print(f"📦 Model resolved at: {MODEL_PATH}")
print("🧠 Loading Tokenizer and Model into cloud GPU...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

# Load tokenizer and model weights cleanly
processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto"
)

print("🚀 Gemma 4 loaded successfully onto the GPU! You are ready to build.")

In [ ]:
from transformers import TextStreamer, StoppingCriteria, StoppingCriteriaList
import json
# 1. Define the structured prompt
messages = [
    {
        "role": "system",
        "content": "You are a strict data-parsing assistant. Extract the requested items, quantities, and units from the user's text into a valid JSON array. Output ONLY the JSON array. Do not include introductory text, explanations, or markdown blocks."
    },
    {
        "role": "user",
        "content": "Habari! Please send 3 crates of soda, 5 bags of sugar, and 2 packets of milk to the shop tomorrow morning."
    },
]

# 2. Process input
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = processor(text=text, return_tensors="pt").to(model.device)

# 3. Initialize the live character-by-character text streamer
streamer = TextStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)

#  Define the specific Gemma end-of-turn tokens
terminators = [
    processor.tokenizer.eos_token_id,
    processor.tokenizer.convert_tokens_to_ids("<end_of_turn>")
]

class JSONArrayComplete(StoppingCriteria):
    def __init__(self, tokenizer, prompt_len):
        self.tokenizer = tokenizer
        self.prompt_len = prompt_len

    def __call__(self, input_ids, scores, **kwargs):
        generated = self.tokenizer.decode(input_ids[0][self.prompt_len:], skip_special_tokens=True)
        start = generated.find("[")
        if start == -1:
            return False
        end = generated.find("]", start)
        if end == -1:
            return False
        try:
            json.loads(generated[start:end + 1])
            return True   # valid array found - stop right here, ignore anything after
        except json.JSONDecodeError:
            return False

prompt_len = inputs["input_ids"].shape[-1]
stopping_criteria = StoppingCriteriaList([JSONArrayComplete(processor.tokenizer, prompt_len)])

print("⏳ Generating structured response from Gemma 4 (Live Stream Below):\n")
print("--- LIVE GENERATED OUTPUT ---")

# 4. Generate with the proper terminators attached
outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.1,
    streamer=streamer,
    repetition_penalty=1.05,    # discourages repeating the same block
    no_repeat_ngram_size=8,    #  hard-blocks repeating any 8-token sequence
    eos_token_id=terminators,     # Forces the model to stop instantly after the JSON
    pad_token_id=processor.tokenizer.pad_token_id,
    stopping_criteria=stopping_criteria,
)

print("\n-----------------------------")
print("✅ Generation Complete!")

## Part B — Duka Akili's grounded backend

Reusing the exact `model` and `processor` loaded above, this is the
[Duka Akili](https://github.com/zackseal89/duka-akili) shop assistant's real
behaviour. It answers only from the shop's own documents (supplied inline here as
the RETRIEVED PASSAGES a search tool would return) and shows the three things the
project is built on: **grounded citation**, **refusal when ungrounded**, and
**flagging a conflict** between the shop's records instead of silently picking one.

In [ ]:
# Duka Akili's real system instruction (from app/agent.py) + a generate helper
# that reuses the model loaded in Part A.
INSTRUCTION = """You are Duka Akili, the knowledge assistant for a small Kenyan retail shop.
You answer questions from the shop's own documents: supplier contracts, the staff
handbook, the pricing and discount policy, and the KRA turnover tax guide. The
RETRIEVED PASSAGES in the user message are the result of searching those documents.

How you must behave:
1. Ground every answer only in the RETRIEVED PASSAGES. Never use general knowledge.
2. Always cite the document and section behind each claim, like
   "supplier_contract_unga_millers.md, section 2. Returns and Damaged Stock".
3. Refuse when ungrounded. If there are no relevant passages, say plainly that the
   shop's documents do not cover it and suggest who to ask. Never guess.
4. Flag conflicts. If two passages give different rules for the same situation, do
   not quietly pick one. Say explicitly that the shop's records disagree, quote
   both, note their effective dates, and recommend the later-dated one while
   advising the owner to update the stale document.
5. Never do money arithmetic yourself.
6. Match the user's language (English or Kiswahili). Keep answers short and
   practical, the way you would explain something across a shop counter."""

def ask(user_text, max_new_tokens=512):
    msgs = [{"role": "system", "content": INSTRUCTION},
            {"role": "user", "content": user_text}]
    text = processor.apply_chat_template(msgs, tokenize=False,
                                         add_generation_prompt=True, enable_thinking=False)
    inputs = processor(text=text, return_tensors="pt").to(model.device)
    n = inputs["input_ids"].shape[-1]
    term = [processor.tokenizer.eos_token_id,
            processor.tokenizer.convert_tokens_to_ids("<end_of_turn>")]
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         repetition_penalty=1.05, no_repeat_ngram_size=8, eos_token_id=term,
                         pad_token_id=processor.tokenizer.pad_token_id or processor.tokenizer.eos_token_id)
    return processor.tokenizer.decode(out[0][n:], skip_special_tokens=True).strip()

print("Helper ready.")

In [ ]:
# CASE 1 - CONFLICT: the genuine three-way disagreement shipped in app/docs/.
CONFLICT = """RETRIEVED PASSAGES:
[1] supplier_contract_unga_millers.md, section 2. Returns and Damaged Stock (effective 2026-03-01)
    "Damaged or torn packaging must be reported to the Supplier within 48 hours of delivery, with photographic evidence via WhatsApp."
[2] staff_handbook.md, section 4.2 Damaged or torn stock (last updated 2025-08-10)
    "If any bags arrive damaged or torn, report it to the shop manager within 7 days of delivery."
[3] supplier_contract_coastal_beverages.md, section 2. Returns and Damaged Stock (effective 2026-05-20)
    "Breakages must be reported to the delivery driver immediately; claims after the driver has left are not honored."

Question: How long do I have to report damaged stock from Unga Millers?"""
print(ask(CONFLICT))

In [ ]:
# CASE 2 - GROUNDED: a single clear source, should answer and cite.
GROUNDED = """RETRIEVED PASSAGES:
[1] supplier_contract_unga_millers.md, section 1. Payment Terms (effective 2026-03-01)
    "Payment must be made via M-Pesa Paybill 400-200 or bank transfer. Cash payments to delivery drivers are not accepted."

Question: How do I pay Unga Millers?"""
print(ask(GROUNDED))

In [ ]:
# CASE 3 - UNGROUNDED: nothing matches, should refuse rather than invent.
REFUSAL = """RETRIEVED PASSAGES:
(none - no passage in the business documents matched this question)

Question: What is the shop's wifi password?"""
print(ask(REFUSAL))

## Optional — serve it live for the ADK app (separate session)

The cells above prove the model in-notebook. To give the deployed **ADK agent**
real function calling, serve the same Kaggle weights with vLLM's `gemma4` tool
parser and expose a URL. This is guarded by `RUN_VLLM_SERVER` and skipped on a
normal Run All — run it in a **fresh** GPU notebook (vLLM pins its own
transformers and conflicts with the git build installed in Part A).

In [ ]:
RUN_VLLM_SERVER = False  # set True in a fresh GPU session to serve the model

if RUN_VLLM_SERVER:
    import subprocess, time, re, urllib.request, kagglehub
    subprocess.run(["pip", "install", "-q", "-U", "vllm", "openai"], check=True)
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
                   "-O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared", shell=True, check=True)

    path = kagglehub.model_download("google/gemma-4/transformers/gemma-4-12b-it")
    subprocess.Popen([
        "vllm", "serve", path, "--served-model-name", "gemma-4-12b-it",
        "--tensor-parallel-size", "2", "--max-model-len", "8192",
        "--enable-auto-tool-choice", "--tool-call-parser", "gemma4",
        "--reasoning-parser", "gemma4", "--gpu-memory-utilization", "0.90",
    ])
    for _ in range(60):
        try:
            urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5); break
        except Exception:
            time.sleep(10)

    tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in tunnel.stdout:
        print(line, end="")
        m = re.search(r"https://[-\w.]+\.trycloudflare\.com", line)
        if m:
            print("\n\nPUBLIC URL:", m.group(0))
            print("On the app:  export MODEL_BACKEND=vllm  VLLM_API_BASE=%s/v1" % m.group(0))
            break
else:
    print("RUN_VLLM_SERVER is False - skipping the live server (see the app's MODEL_BACKEND=vllm path).")